## ML_1041: Gradient Accumulation

**Difficulty**: Easy | **TorchCode**: #31

### Problem
Implement gradient accumulation to simulate a large batch by processing multiple micro-batches before a single optimizer step. Scale each micro-batch loss by `1/n` so gradients match a full-batch update.

### Formula
$$\mathcal{L}_{\text{total}} = \frac{1}{n}\sum_{i=1}^n \mathcal{L}(x_i, y_i), \quad \text{optimizer.step once after all micro-batches}$$

In [ ]:
import torch
import torch.nn as nn

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    optimizer.zero_grad()
    total_loss = 0.0
    n = len(micro_batches)
    for x, y in micro_batches:
        loss = loss_fn(model(x), y) / n
        loss.backward()
        total_loss += loss.item()
    optimizer.step()
    return total_loss

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Matches full batch update ---
torch.manual_seed(0)
model = nn.Linear(4, 2, bias=False)
model_ref = nn.Linear(4, 2, bias=False)
model_ref.load_state_dict(model.state_dict())
loss_fn = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
opt_ref = torch.optim.SGD(model_ref.parameters(), lr=0.1)
x1, y1 = torch.randn(2, 4), torch.randn(2, 2)
x2, y2 = torch.randn(2, 4), torch.randn(2, 2)
accumulated_step(model, opt, loss_fn, [(x1, y1), (x2, y2)])
opt_ref.zero_grad()
loss_fn(model_ref(torch.cat([x1, x2])), torch.cat([y1, y2])).backward()
opt_ref.step()
assert torch.allclose(model.weight.data, model_ref.weight.data, atol=1e-5)

# --- Test 2: Returns loss value ---
model2 = nn.Linear(4, 2)
opt2 = torch.optim.SGD(model2.parameters(), lr=0.01)
loss = accumulated_step(model2, opt2, nn.MSELoss(), [(torch.randn(2, 4), torch.randn(2, 2))])
assert isinstance(loss, float) and loss > 0

# --- Test 3: Parameters actually update ---
model3 = nn.Linear(4, 2)
opt3 = torch.optim.SGD(model3.parameters(), lr=0.1)
w_before = model3.weight.data.clone()
accumulated_step(model3, opt3, nn.MSELoss(), [(torch.randn(2, 4), torch.randn(2, 2))])
assert not torch.equal(model3.weight.data, w_before)

print('All tests passed!')